# **DistilBERT Model**

In [ ]:
!pip install transformers datasets sentencepiece

In [ ]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

dataset = load_dataset("stanfordnlp/imdb")
train_dataset, test_dataset = dataset['train'], dataset['test']

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def preprocess_function(examples):
  return tokenizer(examples["text"], truncation=True, padding=True, max_length=512)

tokenized_train = train_dataset.map(preprocess_function, batched=True)
tokenized_test = test_dataset.map(preprocess_function, batched=True)

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)

In [ ]:
training_args = TrainingArguments(
    output_dir='./results',
    eval_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=10
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    processing_class=tokenizer
)

trainer.train()

In [ ]:
results = trainer.evaluate()
print(f"Evalution Results: {results}")

In [ ]:
new_review = "This movie was fantastic! I loved every minute of it."
inputs = tokenizer(new_review, return_tensors="pt", truncation=True, padding=True, max_length=512)
inputs = {
    key: value.to(device) for key, value in inputs.items()
}
outputs = model(**inputs)
predicted_class = torch.argmax(outputs.logits, dim=1)
print("Positive" if predicted_class.item() == 1 else "Negative")

# **T5 (Text-to-Text Transfer Transformer)**

In [ ]:
from transformers import T5Tokenizer, T5ForConditionalGeneration

model_name = "t5-small"
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

In [ ]:
input_text = "summarize: The Text-to-Text Transfer Transformer (T5) is a model developed by Google. It treats NLP problems as text generation tasks."
inputs = tokenizer(input_text, return_tensors="pt")

In [ ]:
summary_ids = model.generate(inputs.input_ids, max_length=150, num_beams=4, early_stopping=True)
output_text = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
print(output_text)

In [ ]:
input_text = "translate English to French: How are you?"
inputs = tokenizer(input_text, return_tensors="pt")
translation_ids = model.generate(inputs.input_ids, max_length=50, num_beams=5, early_stopping=True)
translation_text = tokenizer.decode(translation_ids[0], skip_special_tokens=True)
print(translation_text)

# **Text Summarization using HuggingFace Model**

Hugging Face → Company & ecosystem
Transformers → Python library
GPT-2, BERT, T5, RoBERTa, Llama, etc. → Models that the Transformers library loads and uses

Think of it like this:

```
Hugging Face
│
├── Transformers (Python Library)
│   │
│   ├── GPT-2
│   ├── BERT
│   ├── RoBERTa
│   ├── DistilBERT
│   ├── T5
│   ├── GPT-Neo
│   ├── BLOOM
│   └── Llama
│
├── Datasets
├── Tokenizers
├── Evaluate
├── PEFT
└── Diffusers
```

In [ ]:
text = """
Artificial Intelligence is transforming industries across the globe.
From healthcare to finance, AI systems are automating processes,
analyzing data and improving decision making.
Organizations are investing heavily in AI research,
though ethical and privacy challenges remain.
"""

input_text = "summarize: " + text

In [ ]:
inputs = tokenizer.encode(
    input_text,
    return_tensors="pt",
    max_length=512,
    truncation=True
)

In [ ]:
summary_ids = model.generate(
    inputs,
    max_length=150,
    min_length=40,
    length_penalty=2.0,
    num_beams=4,
    early_stopping=True
)

In [ ]:
summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
print(summary)

# **GPT-2 with Pipeline**

In [ ]:
!pip install accelerate evaluate

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="openai-community/gpt2",
    device=0 if torch.cuda.is_available() else -1
)

In [ ]:
prompt = "Artificial intelligence will change education because"

output = generator(
    prompt,
    max_new_tokens=60,
    do_sample=True,
    temperature=0.8,
    top_p =0.9,
    num_return_sequences=1
)

print(output[0]["generated_text"])

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("openai-community/gpt2")

text = "I want to learn Hugging Face Transformers."
tokens = tokenizer.tokenize(text)
ids = tokenizer.encode(text)

print("Tokens:", tokens)
print("Token IDs:", ids)
print("Decoded:", tokenizer.decode(ids))

In [ ]:
from transformers import AutoModelForCausalLM

model = "openai-community/gpt2"
tokenizer = AutoTokenizer.from_pretrained(model)
model = AutoModelForCausalLM.from_pretrained(model)
model.to(device)

In [ ]:
prompt = "Machine learning is useful because"

inputs = tokenizer(prompt, return_tensors="pt").to(device)
outputs = model.generate(
    **inputs,
    max_new_tokens=60,
    do_sample=True,
    temperature=0.8,
    top_p=0.9,
    pad_token_id=tokenizer.eos_token_id
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [ ]:
def generate_text(prompt, temperature=0.8, top_p=0.9, max_new_tokens=60):
  inputs = tokenizer(prompt, return_tensors="pt").to(device)

  outputs = model.generate(
      **inputs,
      max_new_tokens=max_new_tokens,
      do_sample=True,
      temperature=temperature,
      top_p=top_p,
      pad_token_id=tokenizer.eos_token_id
  )

  return tokenizer.decode(outputs[0], skip_special_tokens=True)

print(generate_text("The future of software engineering is", temperature=0.7))

In [ ]:
from datasets import Dataset

model_name = "openai-community/gpt2"
texts = [
    "Question: What is AI? Answer: AI means machines performing tasks that require intelligence.",
    "Question: What is machine learning? Answer: Machine learning allows computers to learn from data.",
    "Question: What is NLP? Answer: NLP means Natural Language Processing.",
    "Question: What is Hugging Face? Answer: Hugging Face provides tools and models for machine learning.",
    "Question: What is GPT-2? Answer: GPT-2 is a transformer model used for text generation."
]

dataset = Dataset.from_dict({"text": texts})
dataset

In [ ]:
tokenizer.pad_token = tokenizer.eos_token

def tokenize_function(examples):
  return tokenizer(
      examples["text"],
      padding="max_length",
      truncation=True,
      max_length=512,
  )

tokenized_dataset = dataset.map(tokenize_function, batched=True)
tokenized_dataset = tokenized_dataset.remove_columns(["text"])
tokenized_dataset.set_format("torch")

In [ ]:
model = AutoModelForCausalLM.from_pretrained(model_name)
model.resize_token_embeddings(len(tokenizer))
model = model.to(device)

In [ ]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

In [ ]:
import sys
from datasets import config

config.TORCHVISION_AVAILABLE = False
if "torchvision" in sys.modules:
    del sys.modules["torchvision"]

training_args = TrainingArguments(
    output_dir="./gpt2-finetuned-demo",
    num_train_epochs=5,
    per_device_train_batch_size=2,
    save_steps=50,
    logging_steps=1,
    learning_rate=5e-5,
    weight_decay=0.01,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)

trainer.train()

In [ ]:
prompt = "Question: What is NLP? Answer:"
inputs = tokenizer(prompt, return_tensors="pt").to(device)
outputs = model.generate(
    **inputs,
    max_new_tokens=60,
    do_sample=True,
    temperature=0.7,
    top_p=0.9,
    pad_token_id=tokenizer.eos_token_id
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [ ]:
save_path = "./my-gpt2-model"

model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

In [ ]:
new_tokenizer = AutoTokenizer.from_pretrained(save_path)
new_model = AutoModelForCausalLM.from_pretrained(save_path).to(device)

prompt = "Question: What is Hugging Face? Answer:"
inputs = new_tokenizer(prompt, return_tensors="pt").to(device)

outputs = new_model.generate(
    **inputs,
    max_new_tokens=50,
    pad_token_id=new_tokenizer.eos_token_id
)

print(new_tokenizer.decode(outputs[0], skip_special_tokens=True))

In [ ]:
from huggingface_hub import notebook_login

notebook_login()
model.push_to_hub("my-gpt2-model")
tokenizer.push_to_hub("my-gpt2-model")